# 大相場銘柄の予測と特徴量重要度の検証

特定銘柄のテーマに関連する銘柄リストを取得し、大相場銘柄（相対リターンが大きい銘柄）の予測モデルを作成します。

## 分析パイプライン
1. **データ準備**: 期間と特定銘柄から同テーマの銘柄リストを取得。時系列特徴量を抽出。正解ラベル（テーマ内平均リターンの1.2倍）を設定。
2. **特徴量エンジニアリング**: 日付ごとのZ-score正規化、PCAでの次元削減。
3. **モデル検証**: Logistic Regression (Lasso) と RandomForest を使用し、ROC-AUCを計算。
4. **出力**: 特徴量重要度を棒グラフで表示。


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

# 日本語フォントの設定（環境に合わせて調整が必要な場合があります）
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Hiragino Maru Gothic Pro', 'Yu Gothic', 'Meiryo', 'Takao', 'IPAexGothic', 'IPAPGothic', 'VL PGothic', 'Noto Sans CJK JP']

# 定数の設定
TARGET_CODE = "1301" # 対象銘柄のコード
START_DATE = "20230101"
END_DATE = "20231231"
FORWARD_DAYS = 20 # 20日後のリターンで判定
OUTPERFORM_THRESHOLD = 1.2 # セクター平均リターンの1.2倍を正解とする

DB_PATH = "../data/stock_data.db"
JSON_PATH = "../data/ticker_dictionary.json"


In [ ]:
def get_theme_stocks(target_code, json_path):
    """特定銘柄から同テーマの銘柄リストを取得する"""
    if not os.path.exists(json_path):
        print(f"Warning: {json_path} does not exist. Using mock data.")
        return [target_code, "1332", "1333"] # Mock data for tests
    
    with open(json_path, 'r', encoding='utf-8') as f:
        ticker_data = json.load(f)
        
    target_tags = set()
    for item in ticker_data:
        if item.get("code") == target_code:
            # Sector_MajorやEnd_Marketなどのタグを取得
            if "layered_tags" in item:
                target_tags.update(item["layered_tags"].get("Sector_Major", []))
                target_tags.update(item["layered_tags"].get("End_Market", []))
            elif "tags" in item:
                target_tags.update(item.get("tags", []))
            break
            
    if not target_tags:
        return [target_code]
        
    theme_stocks = []
    for item in ticker_data:
        item_tags = set()
        if "layered_tags" in item:
            item_tags.update(item["layered_tags"].get("Sector_Major", []))
            item_tags.update(item["layered_tags"].get("End_Market", []))
        elif "tags" in item:
            item_tags.update(item.get("tags", []))
            
        if target_tags & item_tags: # 共通のタグがある場合
            theme_stocks.append(item.get("code"))
            
    return list(set(theme_stocks))

theme_stock_codes = get_theme_stocks(TARGET_CODE, JSON_PATH)
print(f"ターゲット銘柄: {TARGET_CODE}")
print(f"抽出された同テーマ銘柄数: {len(theme_stock_codes)}")


In [ ]:
def load_stock_data(codes, start_date, end_date, db_path):
    """データベースから時系列特徴量を取得する"""
    if not os.path.exists(db_path):
        print(f"Warning: {db_path} does not exist. Using mock data.")
        # ダミーデータの生成
        dates = pd.date_range(start="2023-01-01", periods=100).strftime('%Y%m%d')
        mock_df = pd.DataFrame(index=pd.MultiIndex.from_product([dates, codes], names=["date", "code"])).reset_index()
        mock_df["close"] = np.random.lognormal(mean=7, sigma=0.5, size=len(mock_df))
        mock_df["vol_inc_pct"] = np.random.randn(len(mock_df))
        mock_df["atr_norm"] = np.abs(np.random.randn(len(mock_df)))
        mock_df["margin_ratio"] = np.abs(np.random.randn(len(mock_df)))
        mock_df["pbr"] = np.abs(np.random.randn(len(mock_df)))
        mock_df["per"] = np.abs(np.random.randn(len(mock_df)))
        return mock_df
        
    conn = sqlite3.connect(db_path)
    
    code_list_str = "','".join(codes)
    
    # daily_prices, daily_trade_indicators, daily_financials などから結合
    query = f"""
    SELECT 
        dp.date, dp.code, dp.close,
        dti.vol_inc_pct, dti.atr_norm, dti.margin_ratio,
        df.pbr_actual as pbr, df.per_forecast as per
    FROM 
        daily_prices dp
    LEFT JOIN daily_trade_indicators dti ON dp.code = dti.code AND dp.date = dti.date
    LEFT JOIN daily_financials df ON dp.code = df.code AND dp.date = df.date
    WHERE 
        dp.code IN ('{code_list_str}')
        AND dp.date >= '{start_date}'
        AND dp.date <= '{end_date}'
    """
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # 欠損値の補間
    df.fillna(method='ffill', inplace=True)
    df.fillna(0, inplace=True) # forward fillで埋まらない分
    
    return df

df_raw = load_stock_data(theme_stock_codes, START_DATE, END_DATE, DB_PATH)
print(f"取得したデータ件数: {len(df_raw)}")


In [ ]:
def create_features_and_labels(df, forward_days, threshold):
    """
    データリーケージ排除のため、T日の特徴量を用いてT+N日後のリターンを予測。
    正解ラベルは「テーマ内平均リターンの1.2倍」を超えるか。
    """
    df = df.copy()
    
    # dateでソートし、codeごとにグループ化
    df.sort_values(['code', 'date'], inplace=True)
    
    # T+N日のリターン計算 (T日の終値に対する T+N日の終値の変化率)
    # これにより、future_returnは「今日(T日)の終値からN日後の終値へのリターン」となる
    df['future_return'] = df.groupby('code')['close'].pct_change(periods=forward_days).shift(-forward_days)
    
    # dateごとにテーマ内平均リターンを計算
    daily_mean_return = df.groupby('date')['future_return'].transform('mean')
    
    # 正解ラベルの定義: テーマ内平均リターンの threshold 倍を超え、かつプラスリターンであること
    df['label'] = ((df['future_return'] > daily_mean_return * threshold) & (df['future_return'] > 0)).astype(int)
    
    # 予測対象となる日付において、X(特徴量)は全て当日(T日)以前の情報に基づいている
    # y(ターゲット)はT+N日の情報を利用している
    # よってデータリーケージは発生していない
    
    # 欠損値削除 (N日分は未来のリターンが計算できないためNaNになる)
    df.dropna(subset=['future_return'], inplace=True)
    
    return df

df_processed = create_features_and_labels(df_raw, FORWARD_DAYS, OUTPERFORM_THRESHOLD)
print(f"ラベル分布: \n{df_processed['label'].value_counts()}")


In [ ]:
# 特徴量のリスト
feature_cols = ['vol_inc_pct', 'atr_norm', 'margin_ratio', 'pbr']
if 'per' in df_processed.columns:
    feature_cols.append('per')

# 日付ごとのZ-score正規化
def zscore_by_date(x):
    if len(x) > 1 and x.std() > 0:
        return (x - x.mean()) / x.std()
    return np.zeros_like(x)

for col in feature_cols:
    df_processed[f'{col}_norm'] = df_processed.groupby('date')[col].transform(zscore_by_date)

norm_feature_cols = [f'{col}_norm' for col in feature_cols]

# PCA（可視化と多重共線性の確認用）
pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_processed[norm_feature_cols])
df_processed['pca1'] = pca_result[:, 0]
df_processed['pca2'] = pca_result[:, 1]

# PCAの可視化
plt.figure(figsize=(10, 6))
sns.scatterplot(x='pca1', y='pca2', hue='label', data=df_processed, alpha=0.6, palette='coolwarm')
plt.title("PCA Projection of Normalized Features")
plt.show()

print(f"PCA寄与率: {pca.explained_variance_ratio_}")


In [ ]:
# 時系列での分割（未来のデータでテストする）
dates = sorted(df_processed['date'].unique())
# 十分なデータがない場合は分割比率を調整するか、モック用にスキップ
if len(dates) > 10:
    train_dates = dates[:int(len(dates)*0.8)]
    test_dates = dates[int(len(dates)*0.8):]
else:
    train_dates = dates
    test_dates = dates

train_df = df_processed[df_processed['date'].isin(train_dates)]
test_df = df_processed[df_processed['date'].isin(test_dates)]

X_train = train_df[norm_feature_cols]
y_train = train_df['label']
X_test = test_df[norm_feature_cols]
y_test = test_df['label']

# 訓練データに正例と負例が両方存在するか確認
if len(y_train.unique()) > 1 and len(y_test.unique()) > 1:
    # ロジスティック回帰 (Lasso / L1)
    lr_model = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, class_weight='balanced')
    lr_model.fit(X_train, y_train)
    
    # Random Forest
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf_model.fit(X_train, y_train)
    
    # 予測確率
    lr_preds = lr_model.predict_proba(X_test)[:, 1]
    rf_preds = rf_model.predict_proba(X_test)[:, 1]
    
    # ROC-AUC評価
    lr_auc = roc_auc_score(y_test, lr_preds)
    rf_auc = roc_auc_score(y_test, rf_preds)
    
    print(f"Logistic Regression ROC-AUC: {lr_auc:.3f}")
    print(f"Random Forest ROC-AUC: {rf_auc:.3f}")
    
    # ROC曲線の描画
    fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_preds)
    fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_preds)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={lr_auc:.3f})')
    plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for Outperform Prediction')
    plt.legend()
    plt.show()
else:
    print("Not enough classes in training/testing data to train models.")
    # モック用のダミーモデル
    lr_model = LogisticRegression(penalty='l1', solver='liblinear', random_state=42)
    # ダミーデータを少し増やしてfitさせる
    lr_model.fit(np.random.randn(10, len(norm_feature_cols)), np.random.randint(0, 2, 10))
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(np.random.randn(10, len(norm_feature_cols)), np.random.randint(0, 2, 10))


In [ ]:
# 特徴量重要度の抽出
# ロジスティック回帰の係数（絶対値）
lr_coef = np.abs(lr_model.coef_[0])
# RandomForestの特徴量重要度
rf_importance = rf_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'LR_Coef_Abs': lr_coef,
    'RF_Importance': rf_importance
})

# ソート
importance_df_lr = importance_df.sort_values(by='LR_Coef_Abs', ascending=False)
importance_df_rf = importance_df.sort_values(by='RF_Importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Logistic Regression
sns.barplot(x='LR_Coef_Abs', y='Feature', data=importance_df_lr, ax=axes[0], color='skyblue')
axes[0].set_title('Logistic Regression (L1) - Absolute Coefficients')
axes[0].set_xlabel('Absolute Coefficient Value')

# Random Forest
sns.barplot(x='RF_Importance', y='Feature', data=importance_df_rf, ax=axes[1], color='lightgreen')
axes[1].set_title('Random Forest - Feature Importances')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()
